# DRCFNet Model - CMU-MOSI Dataset

Dynamic Reconstructive Context Fusion Network for Sentiment Analysis

In [ ]:
!git clone https://github.com/M-Jafarkhani/Multimodal-Emotion-Recognition

In [ ]:
import gdown

file_id = "1szKIqO0t3Be_W91xvf6aYmsVVUa7wDHU"
destination = "mosi_raw.pkl"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}", destination, quiet=False)

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.append('/content/Multimodal-Emotion-Recognition/src')

from loader import get_dataloader
from models.drcfnet import DRCFNet
from training.drcfnet_loss import DRCFNetLoss
from training.drcfnet_train import train, test
from evaluation.performance import eval_affect

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

In [ ]:
# Load MOSI Data
train_data, valid_data, test_data = get_dataloader(
    '/content/mosi_raw.pkl', 
    max_pad=True, 
    max_seq_len=50
)

In [ ]:
# Initialize Model and Loss
model = DRCFNet(dim_v=35, dim_a=74, dim_t=300, d=128, n_heads=4, dropout=0.2).to(device)
criterion = DRCFNetLoss(lambda_orth=0.01, lambda_contrastive=0.01, task_weight=1.0).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

In [ ]:
# Train the model
model = train(
    model=model,
    train_loader=train_data,
    valid_loader=valid_data,
    criterion=criterion,
    optimizer=optimizer,
    epochs=100,
    device=device
)

In [ ]:
# Test the model
test_loss, preds, labels = test(
    model=model,
    dataloader=test_data,
    criterion=criterion,
    device=device,
    return_preds=True
)

# Evaluate metrics
prede = []
oute = preds.cpu().numpy().tolist()
for i in oute:
    if i[0] > 0:
        prede.append(1)
    elif i[0] < 0:
        prede.append(-1)
    else:
        prede.append(0)
pred_classes = torch.LongTensor(prede)

accs = eval_affect(labels, pred_classes)
acc2 = eval_affect(labels, pred_classes, exclude_zero=False)

print(f"Recall: {accs*100:.4f}% | Total Accuracy: {acc2*100:.4f}%")